In [1]:
import sys, importlib
import numpy as np
import pandas as pd
sys.path.append("../src")

df = pd.read_pickle("../data/processed/itsm_clean.pkl")

## store the dates in one place.

In [2]:
%%writefile ../src/project_config.py
START = "2013-10-01"
TEST_START = "2014-02-01"
CUT = "2014-03-17"
HIGH = [1, 2]

Overwriting ../src/project_config.py


## import it and build the split.

In [3]:
sys.modules.pop("project_config", None)
importlib.invalidate_caches()
from project_config import START, TEST_START, CUT, HIGH

mw = df[(df["Open_Time"] >= START) & (df["Open_Time"] < CUT) & df["Priority"].notna()]
mw = mw.sort_values("Open_Time").reset_index(drop=True)
mw["High"] = mw["Priority"].isin(HIGH).astype(int)

train = mw[mw["Open_Time"] < TEST_START]
test = mw[mw["Open_Time"] >= TEST_START]
print(len(train), train["High"].sum(), len(test), test["High"].sum())

30845 541 10229 115


## add open-time features

In [4]:
mw = mw.drop(columns=["month"], errors="ignore")
mw["hour"] = mw["Open_Time"].dt.hour
mw["weekday"] = mw["Open_Time"].dt.day_of_week
train = mw[mw["Open_Time"] < TEST_START]
test = mw[mw["Open_Time"] >= TEST_START]

## check the categorical columns

In [5]:
cats = ["CI_Name", "CI_Cat", "CI_Subcat", "WBS", "Category"]
for c in cats:
    unseen = (~test[c].isin(train[c].unique())).mean() * 100
    print(f"{c}: {train[c].nunique()} values in train, {unseen:.1f}% of test rows have a value never seen in train")

CI_Name: 2471 values in train, 4.9% of test rows have a value never seen in train
CI_Cat: 13 values in train, 0.0% of test rows have a value never seen in train
CI_Subcat: 58 values in train, 0.0% of test rows have a value never seen in train
WBS: 255 values in train, 0.6% of test rows have a value never seen in train
Category: 4 values in train, 0.0% of test rows have a value never seen in train


In [6]:
print("Test High rate (a random model's average precision):", round(test["High"].mean(), 4))
print("Accuracy of always predicting not-High:", round(1 - test["High"].mean(), 4))

import os
os.makedirs("../data/processed", exist_ok=True)
train.to_pickle("../data/processed/track1_train.pkl")
test.to_pickle("../data/processed/track1_test.pkl")

Test High rate (a random model's average precision): 0.0112
Accuracy of always predicting not-High: 0.9888


## How to read it: do the features carry signal? (aggregates only)

1. Rates by Category and CI_Cat: if they vary a lot (say 0.5% in one group and 5% in another), the asset fields carry signal.
2. The two CI shares: if the 30+ CIs cover most of the High tickets, the “infrequent” bucket loses little. If they cover few, CI_Name needs another approach.
3. Hour and weekday: a flat result means these add little.

In [7]:
for c in ["Category", "CI_Cat"]:
    g = train.groupby(c)["High"].agg(rows="size", high="sum")
    g["rate_%"] = (g["high"] / g["rows"] * 100).round(2)
    print(g.sort_values("rows", ascending=False).head(8), "\n")

g = train.groupby("CI_Name")["High"].agg(rows="size", high="sum")
big = g["rows"] >= 30
print("CI_Names with at least one High:", int((g["high"] > 0).sum()), "of", len(g))
print("CI_Names with 30+ train rows:", int(big.sum()))
print("Share of train rows on those CIs:", round(g.loc[big, "rows"].sum() / g["rows"].sum() * 100, 1), "%")
print("Share of train High on those CIs:", round(g.loc[big, "high"].sum() / g["high"].sum() * 100, 1), "%")

print("High rate % by hour:", train.groupby("hour")["High"].mean().mul(100).round(1).to_dict())
print("High rate % by weekday:", train.groupby("weekday")["High"].mean().mul(100).round(1).to_dict())

                          rows  high  rate_%
Category                                    
incident                 24416   537    2.20
request for information   6421     4    0.06
complaint                    7     0    0.00
request for change           1     0    0.00 

                    rows  high  rate_%
CI_Cat                                
application        21779   143    0.66
subapplication      5178    23    0.44
computer            2516   345   13.71
storage              424     5    1.18
hardware             255     2    0.78
software             200     1    0.50
displaydevice        143     0    0.00
officeelectronics    106     0    0.00 

CI_Names with at least one High: 358 of 2471
CI_Names with 30+ train rows: 139
Share of train rows on those CIs: 83.5 %
Share of train High on those CIs: 23.7 %
High rate % by hour: {0: 2.3, 1: 0.0, 2: 0.0, 3: 0.0, 4: 33.3, 5: 23.8, 6: 24.3, 7: 5.4, 8: 1.9, 9: 2.0, 10: 1.6, 11: 1.6, 12: 1.7, 13: 1.6, 14: 1.4, 15: 1.0, 16: 1.7, 17: 1.9

In [8]:
%%writefile ../src/features.py
asset_cols = ["CI_Cat", "CI_Subcat", "WBS", "Category"]
ci_name    = ["CI_Name"]
time_feats = ["hour", "weekday"]
matrix_cols = ["Impact", "Urgency"]

post_event = ["Status", "Resolved_Time", "Close_Time", "Reopen_Time", "Has_Reopen_Time",
              "Handle_Time_hrs", "Elapsed_hrs", "Closure_Code"]
unsure_timing = ["No_of_Reassignments", "No_of_Related_Interactions", "No_of_Related_Incidents",
                 "No_of_Related_Changes", "Has_Related_Change", "KB_number"]
identifiers = ["Incident_ID", "Related_Interaction", "Related_Change"]

set_a = asset_cols + time_feats + matrix_cols   # includes Impact and Urgency
set_b = asset_cols + time_feats                 # the honest test

Overwriting ../src/features.py


In [9]:
sys.modules.pop("features", None)
importlib.invalidate_caches()
from features import *

known = set(asset_cols + ci_name + time_feats + matrix_cols + post_event
            + unsure_timing + identifiers + ["Priority", "High", "Open_Time"])
print(sorted(set(train.columns) - known))

[]
